## Implementation of a simple form of LLM
#### (without any additional enhancement on query / retrieved docs / routing)

In [1]:
from chromadb.config import Settings
from chromadb import Client
from langchain.vectorstores import Chroma
import chromadb

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from typing_extensions import List, TypedDict
from langgraph.graph import START, StateGraph

import os, re
from datetime import datetime

date = datetime.today().strftime('%Y-%m-%d')

# Initialize Langsmith
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGSMITH_PROJECT"] = f"[{date}] VAA - Basic LLM Testing"

# Initialize LLM
llm = ChatOllama(model="deepseek-r1:8b", validate_model_on_init=True, temperature=0.6)
emb = OllamaEmbeddings(model="bge-m3:567m")

In [2]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "polyu_eee_document" if not SINGLE else "vaa_documents"

# Initialize retriever for queries
client = Client(Settings())
client = chromadb.PersistentClient(path="../Code_Extraction/chroma_db")

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=emb
)

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_27487/3522301744.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


In [3]:
# Defining the class structure for the LLM
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

# The LLM prompt
LLM_prompt = \
    """
    You are an professional academic advisor in The Hong Kong Polytechnic University, please adhere to the following rules:
    1. Be affirm, avoid saying "may", "maybe", or anything similar,
    2. Say no if you cannot answer the question, do not fabricate factually-false answer,
    3. Provide advice to the student if necessary.

    Now, please use the following context to answer the student's question.
    Remember to thank the user at the end and ask if there are any more enquiry.

    *Context*:
    {context}

    *Student's Question*:
    {question}

    Helpful Answer:
    """
prompt = PromptTemplate.from_template(LLM_prompt)

# Functions for document retrieval based on cos-sim
def retrieve(state: State):
    retrieved_docs = vectorStore.similarity_search(state["question"], k=4)
    return {"context": retrieved_docs}

# Functions for constructing the final LLM prompt
def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}

# Functions for graph building (a process sequence)
def graph_building():
    global graph
    graph_builder = StateGraph(State).add_sequence([retrieve, generate])
    graph_builder.add_edge(START, "retrieve")
    graph = graph_builder.compile()

graph_building()

In [4]:
query = \
"What is the graduation requirement for my programme (BEng Scheme in Electrical Engineering)?"

dataset = []
print(f"Generating {query}")
result = graph.invoke({"question": query})
dataset.append(
    {
        "user_input": result['question'],
        "retrieved_contexts": [doc.page_content for doc in result['context']],  # Extract text from Documents
        "response": result['answer'],
    }
)
print(f"Answer:\n{re.sub(r"<think>.*?</think>", "", result['answer'], flags=re.DOTALL).strip()}")

Generating What is the graduation requirement for my programme (BEng Scheme in Electrical Engineering)?
Answer:
根据提供的信息，**BEng Scheme in Electrical Engineering**（电气工程学士课程）的毕业要求如下：

1. **学分要求**：
   - **主修课程学分**：84 学分（其中必修课程 75 学分，选修课程 9 学分）。
   - **培训要求学分**：11 学分。
   - **通识教育学分**：27 学分。
   - **自由选修学分**：6 学分。
   - **副修课程要求**（如修读人工智能与数据科学副修）：36 学分。

2. **成绩要求**：
   - 学生需维持累积 GPA 至少为 **2.70**，方可申请副修课程。
   - 若学生 GPA 低于 **1.70**，将被置于学术预警状态，需在后续学期中努力提高成绩，恢复至 **1.70** 或以上方可毕业。

3. **其他要求**：
   - 学生需完成所有课程及培训要求。
   - 若有副修课程，需获得相关院系的批准。

希望以上信息能帮助你更好地规划学业。如有进一步疑问，欢迎随时咨询！感谢你的提问，祝学业顺利！

如有更多疑问，请随时提出 😊
